
# Projeto Referência G2 — Dashboard Executivo de Vendas no Brasil

## Disciplina: Linguagem de Programação — Análise e Visualização de Dados com Python

Este notebook apresenta a etapa de análise exploratória e preparação dos dados do projeto-referência da G2.

O objetivo é demonstrar como sair de uma base de dados bruta e chegar a indicadores, gráficos e interpretações que posteriormente serão usados em um dashboard Streamlit.



# 1. Problema de negócio

Uma empresa de varejo atua em diferentes UFs, canais de venda, categorias e segmentos.

A gestão deseja responder:

- Qual é a receita total?
- Qual é o lucro total?
- Qual é a margem de lucro?
- Qual canal tem melhor desempenho?
- Qual categoria gera maior receita?
- Quais UFs concentram maior volume financeiro?
- Há tendência de crescimento ou queda ao longo do tempo?

Essas perguntas orientam toda a análise.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


# 2. Leitura da base de dados

In [ ]:

df = pd.read_csv("../dados/vendas_brasil.csv")

df.head()


# 3. Check-up inicial da base

In [ ]:

print("Dimensões da base:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nDuplicidades:")
print(df.duplicated().sum())


# 4. Preparação dos dados

In [ ]:

df["data"] = pd.to_datetime(df["data"], errors="coerce")

df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["ano_mes"] = df["data"].dt.to_period("M").astype(str)

df["margem_lucro"] = df["lucro"] / df["receita"]
df["ticket_medio"] = df["receita"] / df["quantidade"]

df.head()



# 5. KPIs principais

Os KPIs foram escolhidos com base em uma visão executiva do negócio:

- Receita total;
- Lucro total;
- Margem de lucro;
- Ticket médio;
- Quantidade total vendida.


In [ ]:

receita_total = df["receita"].sum()
lucro_total = df["lucro"].sum()
margem_lucro = lucro_total / receita_total
ticket_medio = df["receita"].sum() / df["quantidade"].sum()
quantidade_total = df["quantidade"].sum()

print(f"Receita total: R$ {receita_total:,.2f}")
print(f"Lucro total: R$ {lucro_total:,.2f}")
print(f"Margem de lucro: {margem_lucro:.2%}")
print(f"Ticket médio: R$ {ticket_medio:,.2f}")
print(f"Quantidade vendida: {quantidade_total:,.0f}")


# 6. Receita e lucro ao longo do tempo

In [ ]:

serie_mensal = (
    df.groupby("ano_mes")[["receita", "lucro"]]
    .sum()
    .reset_index()
    .sort_values("ano_mes")
)

serie_mensal.head()


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))

sns.lineplot(data=serie_mensal, x="ano_mes", y="receita", marker="o", label="Receita", ax=ax)
sns.lineplot(data=serie_mensal, x="ano_mes", y="lucro", marker="o", label="Lucro", ax=ax)

ax.set_title("Evolução Mensal da Receita e do Lucro")
ax.set_xlabel("Ano-Mês")
ax.set_ylabel("Valor")
ax.tick_params(axis="x", rotation=45)

plt.show()


# 7. Desempenho por canal

In [ ]:

desempenho_canal = (
    df.groupby("canal")
    .agg(
        receita_total=("receita", "sum"),
        lucro_total=("lucro", "sum"),
        quantidade_total=("quantidade", "sum")
    )
    .reset_index()
)

desempenho_canal["margem_lucro"] = desempenho_canal["lucro_total"] / desempenho_canal["receita_total"]

desempenho_canal.sort_values("receita_total", ascending=False)


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=desempenho_canal.sort_values("receita_total", ascending=False),
    x="canal",
    y="receita_total",
    ax=ax
)

ax.set_title("Receita Total por Canal")
ax.set_xlabel("Canal")
ax.set_ylabel("Receita")
ax.tick_params(axis="x", rotation=30)

plt.show()


# 8. Desempenho por categoria

In [ ]:

receita_categoria = (
    df.groupby("categoria")["receita"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

receita_categoria


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(data=receita_categoria, x="categoria", y="receita", ax=ax)

ax.set_title("Receita por Categoria")
ax.set_xlabel("Categoria")
ax.set_ylabel("Receita")
ax.tick_params(axis="x", rotation=45)

plt.show()


# 9. Desempenho por UF

In [ ]:

receita_uf = (
    df.groupby("uf")["receita"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

receita_uf.head()


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))

sns.barplot(data=receita_uf, x="uf", y="receita", ax=ax)

ax.set_title("Receita por UF")
ax.set_xlabel("UF")
ax.set_ylabel("Receita")

plt.show()


# 10. Persistência com SQLAlchemy e SQLite

In [ ]:

engine = create_engine("sqlite:///../database/vendas_brasil.sqlite")

df.to_sql("vendas", engine, if_exists="replace", index=False)

print("Tabela vendas criada no banco SQLite.")


In [ ]:

consulta = '''
SELECT canal,
       SUM(receita) AS receita_total,
       SUM(lucro) AS lucro_total,
       SUM(lucro) / SUM(receita) AS margem_lucro
FROM vendas
GROUP BY canal
ORDER BY receita_total DESC
'''

pd.read_sql(consulta, engine)



# 11. Conclusão executiva

A análise mostra como uma base de vendas pode ser transformada em um produto analítico com valor gerencial.

O projeto permite identificar:

- quais canais concentram mais receita;
- quais canais são mais rentáveis;
- quais categorias têm maior impacto financeiro;
- quais UFs apresentam maior desempenho;
- como receita e lucro evoluem ao longo do tempo.

Essas informações ajudam gestores a tomar decisões sobre investimento, canais de venda, mix de produtos e estratégia regional.
